**Part 2 · LangChain** · step 8 of 25  
[← 07 Structured output](07_structured_output.ipynb) · [🏠 Roadmap](../README.md) · [09 Tools & tool calling →](09_tools_and_tool_calling.ipynb)

# 08 · Conversation History — memory done by hand

LLMs are **stateless**: every call only sees the messages you send in that call.
To build a chatbot you keep a list of messages and append to it after every turn.

```
[System] → [Human 1] → [AI 1] → [Human 2] → [AI 2] → ...   (resend the whole list every turn)
```

Later you'll see the automatic way: an agent/graph **checkpointer** + `thread_id`
(notebook 11).

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage

chat_model = init_chat_model("gpt-4o")

## Scripted conversation (runs without typing)

In [ ]:
def chat(history: list, user_text: str) -> str:
    """Send one user turn and record both sides in ``history``.

    Args:
        history: The running list of messages. It is mutated in place.
        user_text: What the user says this turn.

    Returns:
        The model's reply text.
    """
    history.append(HumanMessage(user_text))
    reply = chat_model.invoke(history)  # the model sees the FULL history
    history.append(reply)               # remember the answer for next time
    return reply.content


history = [SystemMessage("You are a helpful AI assistant")]
print(chat(history, "Hi, I'm Ujjawal and I live in Delhi."))
print(chat(history, "Where do I live?"))  # works only because we resent the history
print(len(history), "messages in history")

## Interactive version
Type messages in the input box; type `exit` to stop.

In [ ]:
chat_session = [SystemMessage("You are a helpful AI assistant")]

while True:
    user_input = input("You: ")
    if user_input.strip().lower() == "exit":
        break
    print("AI:", chat(chat_session, user_input))
    print("*" * 10)

## Revision notes
- The model is stateless → **you** must resend history every call.
- History grows forever → long chats get expensive and eventually exceed the context window.
  Solutions: trimming, or summarisation (`SummarizationMiddleware`, notebook 13).

---
✅ **Done with 08 Conversation history.** Next: [09 Tools & tool calling →](09_tools_and_tool_calling.ipynb)  
[🏠 Roadmap](../README.md)